In [ ]:
import os, eda_analysis

cfg = eda_analysis.EdaConfig(family="arms/outcomes")
S   = eda_analysis.notebook_setup(cfg)

# `arms/outcomes` -- what every arm scored

**The question.** For each arm on disk, at each model state, on each instrument: what did the
grader give it, how did that move across iterations, and where did it end up relative to the
untrained base policy?

**The axis.** `iteration` here is the MODEL STATE (`model_iter_<N>`), which is labelled by the
policy that GENERATED those conversations -- so `iteration = 0` is the untrained base and `N`
training iterations produce `N + 1` states. It is *not* the training-iteration index; the two
are off by one.

**The pairing unit.** Every score is one conversation with one of the 96 fixed personas, and a
persona is the same client in every arm and every iteration. Nothing on this page is paired
(these are descriptives); the paired contrasts live in `lookahead/reward` and `method/contrast`.

**The graders.** Every table and figure names its grader. The default judge is the same local
model that served the training oracle, so it is *not* held out; a second judge, if the lake holds
one, is. They are reported side by side and **never averaged** -- one is train and one is test,
they do not share a scale, and a mean over them applies a silent model-dependent shrinkage.

**Orientation.** Eight instruments are higher-is-better. **MICI is lower-is-better** -- it counts
MI-INCONSISTENT therapist behaviour -- so every ranking below is taken over
`sign_of(metric) * score` and every "gain" column is signed so that **positive always means
better**, on every metric.

**Final AND best.** Both endpoints are reported for every arm. An arm that peaked and then
regressed is flattered by the best-state row and not by the final-state row, and quoting only one
of them chooses the answer; `past_peak = True` marks exactly the arms where the two disagree.

**What this family does not claim.** Overlapping CI bands are not a test. The bands are bootstrap
intervals ACROSS the 96 personas, and persona variance dominates -- a paired contrast removes it,
so two heavily overlapping bands are perfectly compatible with a decisive within-persona
difference. Read differences off the contrast families, never off these panels.

In [ ]:
# ---------------------------------------------------------------------------
# Imports, both graders, and the guards that let this notebook render with NO DATA
# on disk -- which is the normal state until the first arm has been generated and
# scored. Every section below degrades to an explicit "no data yet" artifact.
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from eda_analysis import constants, data, exports, plotting, stats

SEED = S.CFG.boot_seed
FOCUS = S.CFG.focus_metric

# Every grader in the lake, in ONE frame with a `judge` column. Never averaged across
# judges; always shown side by side.
ALL = data.scores_by_judge(S.ARMS, rep=S.CFG.judge_rep, attach_persona=S.CFG.attach_persona)
if ALL.empty and not S.SCORES.empty:
    ALL = S.SCORES
HAVE = not ALL.empty

JUDGES = sorted(ALL["judge"].unique()) if HAVE else [S.JUDGE]
METRICS = [m for m in constants.METRIC_ORDER if HAVE and m in set(ALL["metric"].unique())]
ARMS = sorted(ALL["arm_label"].unique()) if HAVE else []
PALETTE = plotting.arm_palette(ARMS) if ARMS else {}

NO_DATA = ("NO DATA YET -- no scored conversations for these arms. Generate an arm, then run "
           "notebooks/scoring/Run_Eval.ipynb; this family re-renders from the score lake.")

# A clean leaf per render: reset_results clears ONLY this family's figures/ + tables/
# (the hand-authored PRESERVE names are out of reach structurally), and provenance is
# then re-written from the frame actually loaded -- both graders, not just the default.
exports.reset_results()
exports.save_provenance(S.CFG, ALL)


def placeholder(name, message=NO_DATA, group=None, caption=None):
    # A figure-shaped marker, so a section that had nothing to plot still appears in the
    # index saying so, instead of leaving a hole a reader has to interpret.
    fig = plt.figure(figsize=(7.6, 1.9))
    ax = fig.add_subplot(111)
    ax.axis("off")
    ax.text(0.5, 0.5, message, ha="center", va="center", fontsize=8.5,
            color="#777777", wrap=True)
    exports.save_fig(fig, name, group=group, caption=caption or message)
    plt.close(fig)


print(f"judges : {JUDGES}")
print(f"arms   : {ARMS or '(none on disk)'}")
print(f"metrics: {METRICS or '(none scored)'}")
print(f"focus  : {FOCUS}   seed: {SEED}")
if not HAVE:
    print(NO_DATA)

## 1. Coverage -- what is actually scored

Before any number is read, this says which (grader, arm, model state) cells exist and how much of
the 96-persona grid each one covers. A partly-scored state is not an error -- it is the normal
condition between a training run and a scoring run -- but it silently shortens every contrast
computed from it, on both sides, so it belongs next to the numbers rather than in a log.

In [ ]:
def coverage(df):
    # One row per (grader, arm, model state): how many personas and instruments landed.
    if df.empty:
        return pd.DataFrame()
    out = (df.groupby(["judge", "arm_label", "iteration"], as_index=False)
             .agg(model_state=("model_state", "first"),
                  metrics=("metric", "nunique"),
                  personas=("persona_id", "nunique"),
                  scored_cells=("score", "count")))
    out["complete_grid"] = out["personas"] == constants.N_PERSONAS
    return (out.rename(columns={"arm_label": "arm"})
               .sort_values(["judge", "arm", "iteration"])
               .reset_index(drop=True))


COV = coverage(ALL)
exports.save_table(
    COV, "coverage",
    caption=("Scored coverage per grader x arm x MODEL STATE (iteration 0 = untrained base). "
             "`personas` is the number of distinct persona_id values present; anything below 96 "
             "means the state is only partly scored, and every paired contrast elsewhere drops "
             "those personas from BOTH sides. `metrics` counts instruments, not items."))
if COV.empty:
    print(NO_DATA)
else:
    partial = COV[~COV["complete_grid"]]
    print(f"{len(COV)} (grader, arm, state) cells; {len(partial)} below the full 96-persona grid")
    display(COV.head(20))

## 2. Descriptives -- mean, spread and a seeded bootstrap CI

One row per (grader, arm, model state, instrument). `ci_lo`/`ci_hi` are a 2,000-resample
percentile bootstrap of the MEAN **across personas**, seeded with `BOOT_SEED` so that
re-rendering unchanged data reproduces the table byte for byte.

The interval describes the spread across the 96 clients. It is not the uncertainty of a
difference between two arms -- that is a paired quantity, and pairing removes exactly the persona
variance these intervals are made of.

In [ ]:
def describe(df):
    # Per (grader, arm, state, metric): n, mean, sd and a seeded bootstrap CI of the mean.
    if df.empty:
        return pd.DataFrame()
    rows = []
    for (judge, arm, state, metric), g in df.groupby(
            ["judge", "arm_label", "iteration", "metric"], sort=True):
        x = pd.to_numeric(g["score"], errors="coerce").to_numpy(dtype=float)
        x = x[~np.isnan(x)]
        lo, hi = stats.bootstrap_ci(x, np.mean, seed=SEED) if x.size else (np.nan, np.nan)
        rows.append({
            "judge": judge, "arm": arm, "iteration": int(state), "metric": metric,
            "instrument": constants.short_label(metric),
            "n": int(x.size),
            "mean": float(np.mean(x)) if x.size else np.nan,
            "sd": float(np.std(x, ddof=1)) if x.size > 1 else np.nan,
            "ci_lo": lo, "ci_hi": hi,
            "sign": int(constants.sign_of(metric)),
        })
    out = pd.DataFrame(rows)
    return out.sort_values(["judge", "metric", "arm", "iteration"]).reset_index(drop=True)


DESC = describe(ALL)
exports.save_table(
    DESC, "descriptives",
    caption=("Per grader x arm x MODEL STATE x instrument: n personas, mean, SD, and a 2,000-"
             "resample percentile bootstrap CI of the mean ACROSS personas (seed=BOOT_SEED). "
             "`sign` is +1 where higher is better and -1 for MICI. These CIs are unpaired: do "
             "not read an arm difference off two of them."))
if DESC.empty:
    print(NO_DATA)
else:
    display(DESC.head(20))

## 3. Trajectories -- every arm on one axis, one panel per instrument

One figure per grader; one panel per instrument; one line per arm, in its stable colour (an arm
is the same colour in every artifact of this EDA). The dotted reference is the **base level**:
the mean over every arm's `model_iter_0`, i.e. the untrained policy on that instrument under that
grader.

The band is the same unpaired bootstrap CI as the table above -- spread across personas, not the
precision of an arm difference.

In [ ]:
for judge in JUDGES:
    sub = ALL[ALL["judge"] == judge] if HAVE else ALL
    name = f"trajectory_{judge}"
    if sub.empty or not METRICS:
        placeholder(name, caption=f"{NO_DATA} (grader {judge})")
        continue
    fig, axes = plotting.grid(len(METRICS), ncols=3)
    for ax, m in zip(axes, METRICS):
        panel = sub[sub["metric"] == m]
        base = panel[panel["iteration"] == 0]["score"].mean()
        plotting.score_trajectory(
            panel, metric=m, metric_col="metric", arm_col="arm_label", palette=PALETTE,
            base_value=base, ax=ax, title=constants.short_label(m),
            xlabel="model state", ylabel="grader score")
        legend = ax.get_legend()
        if legend is not None and ax is not axes[0]:
            legend.remove()
    exports.save_fig(
        fig, name,
        caption=(f"Mean score by MODEL STATE, one panel per instrument, grader {judge}. "
                 f"State 0 is the untrained base; the dotted line is the base level pooled over "
                 f"arms. Bands are unpaired 95% bootstrap CIs across the 96 personas "
                 f"(seed=BOOT_SEED) -- overlap between two bands is NOT evidence of no "
                 f"difference. MICI is lower-is-better."))
    plt.close(fig)
    print(f"trajectory rendered for grader {judge}")

## 4. Leaderboard -- final state and best state, side by side

For every (grader, instrument, arm):

* `final_*` is the arm's **last** trained model state;
* `best_*` is the state maximising `sign_of(metric) * mean`, i.e. the best checkpoint *on that
  instrument for that grader*;
* `gain_final` / `gain_best` are signed against the base so that **positive is always better**,
  MICI included;
* `past_peak = True` means the best state is not the final state -- the arm regressed after its
  peak, and the two endpoints tell different stories.

Ranks are computed on the oriented value, so MICI ranks the right way round without a special
case at the call site.

In [ ]:
def leaderboard(desc):
    # Final-state and best-state endpoints per (grader, instrument, arm), both signed as gains.
    if desc.empty:
        return pd.DataFrame()
    rows = []
    for (judge, metric, arm), g in desc.groupby(["judge", "metric", "arm"], sort=True):
        g = g.sort_values("iteration")
        sign = int(constants.sign_of(metric))
        base = g[g["iteration"] == 0]["mean"]
        base_mean = float(base.iloc[0]) if len(base) else np.nan
        trained = g[g["iteration"] > 0]
        if trained.empty:
            continue
        oriented = sign * trained["mean"].to_numpy(dtype=float)
        if np.all(np.isnan(oriented)):
            continue
        final = trained.iloc[-1]
        best = trained.iloc[int(np.nanargmax(oriented))]
        rows.append({
            "judge": judge, "metric": metric, "instrument": constants.short_label(metric),
            "arm": arm, "sign": sign, "n_states": int(len(trained)),
            "base_mean": base_mean,
            "final_state": int(final["iteration"]), "final_mean": float(final["mean"]),
            "best_state": int(best["iteration"]), "best_mean": float(best["mean"]),
            "gain_final": sign * (float(final["mean"]) - base_mean),
            "gain_best": sign * (float(best["mean"]) - base_mean),
            "past_peak": int(best["iteration"]) != int(final["iteration"]),
        })
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out["oriented_final"] = out["sign"] * out["final_mean"]
    out["oriented_best"] = out["sign"] * out["best_mean"]
    grp = out.groupby(["judge", "metric"])
    # Ranked on the ORIENTED value, so MICI ranks the right way round with no special case.
    # Int64 (nullable) rather than int: a cell whose mean is NaN has no rank, and a plain
    # astype(int) on that raises rather than leaving the gap visible.
    out["rank_final"] = grp["oriented_final"].rank(ascending=False, method="min").astype("Int64")
    out["rank_best"] = grp["oriented_best"].rank(ascending=False, method="min").astype("Int64")
    cols = ["judge", "metric", "instrument", "arm", "rank_final", "rank_best", "base_mean",
            "final_state", "final_mean", "gain_final", "best_state", "best_mean", "gain_best",
            "past_peak", "n_states", "sign"]
    return out[cols].sort_values(["judge", "metric", "rank_final"]).reset_index(drop=True)


LB = leaderboard(DESC)
exports.save_table(
    LB, "leaderboard",
    caption=("Final-state AND best-state endpoint per grader x instrument x arm. `gain_*` is "
             "signed by sign_of(metric) against the arm's own base (state 0), so POSITIVE IS "
             "BETTER on every instrument, MICI included; ranks are on the oriented mean. "
             "`past_peak` marks an arm whose best state is not its last -- reporting only one "
             "endpoint for those arms chooses the answer."))
exports.save_table(
    LB[LB["metric"] == FOCUS] if not LB.empty else LB, "leaderboard_focus",
    caption=(f"The leaderboard restricted to the training-reward axis ({FOCUS}) -- the metric "
             f"the policy was actually optimized against. Same signing and ranking rules."))
if LB.empty:
    print(NO_DATA)
else:
    peaked = LB[LB["past_peak"]]
    print(f"{len(LB)} endpoint rows; {len(peaked)} (grader, instrument, arm) cells are PAST PEAK")
    display(LB[LB["metric"] == FOCUS])

## 5. Endpoint distributions -- the spread behind each headline mean

The per-persona distribution at each arm's own **final** state and at its own **best** state, on
the training-reward axis. The diamond is the mean with an unpaired bootstrap CI; the box shows the
median, which is not the statistic any table here reports, which is why both are drawn.

The same 96 personas appear in every box, so two boxes overlapping heavily is entirely compatible
with a large, consistent within-persona difference.

In [ ]:
def endpoint_frame(df, lb, judge, metric, state_col):
    # Per-persona rows at each arm's OWN endpoint state (final or best) -- arms peak at
    # different iterations, so a single shared state would not be either endpoint.
    empty = df.iloc[0:0]
    if df.empty or lb.empty:
        return empty
    sel = lb[(lb["judge"] == judge) & (lb["metric"] == metric)][["arm", state_col]]
    parts = []
    for _, r in sel.iterrows():
        parts.append(df[(df["judge"] == judge) & (df["arm_label"] == r["arm"])
                        & (df["metric"] == metric) & (df["iteration"] == int(r[state_col]))])
    return pd.concat(parts, ignore_index=True) if parts else empty


for judge in JUDGES:
    for state_col, tag in (("final_state", "final"), ("best_state", "best")):
        name = f"endpoint_{tag}_{judge}"
        frame = endpoint_frame(ALL, LB, judge, FOCUS, state_col)
        if frame.empty:
            placeholder(name, caption=f"{NO_DATA} (grader {judge}, {tag} state, {FOCUS})")
            continue
        base_level = ALL[(ALL["judge"] == judge) & (ALL["metric"] == FOCUS)
                         & (ALL["iteration"] == 0)]["score"].mean()
        # arm_distribution overlays a seaborn stripplot, whose JITTER draws from numpy's global
        # RNG and accepts no seed argument -- so without this, two renders of identical data
        # produce different PNGs and every tracked figure churns in git. Its bootstrap CI is
        # already seeded inside plotting; the jitter is the one draw that callsite cannot reach.
        np.random.seed(SEED)
        fig = plotting.arm_distribution(
            frame, metric=FOCUS, metric_col="metric", arm_col="arm_label", palette=PALETTE,
            title=f"{constants.short_label(FOCUS)} at each arm's {tag} state ({judge})",
            ylabel="grader score")
        plotting.add_base_line(fig.axes[0], base_level, label="base")
        exports.save_fig(
            fig, name,
            caption=(f"Per-persona {FOCUS} at each arm's OWN {tag} model state, grader {judge}. "
                     f"Box = median, diamond = mean with an unpaired 95% bootstrap CI "
                     f"(seed=BOOT_SEED), dotted line = the untrained base level. The same 96 "
                     f"personas are in every box, so overlap here does not bound a paired "
                     f"difference."))
        plt.close(fig)
print("endpoint distributions rendered")

## 6. Number ledger and index

A citable ledger of the few numbers this family is actually quoted for, each with the table it
came from, followed by the index refresh that ends every family notebook.

In [ ]:
values = {
    "coverage.judges": {"value": JUDGES, "source": "tables/coverage.md",
                        "note": "graders present in the score lake"},
    "coverage.arms": {"value": ARMS, "source": "tables/coverage.md",
                      "note": "arm display labels; key on experiment_name for data"},
    "coverage.metrics": {"value": METRICS, "source": "tables/coverage.md", "note": ""},
    "coverage.scored_rows": {"value": int(len(ALL)), "source": "tables/coverage.md",
                             "note": "one row per (arm, state, persona, instrument, grader)"},
    "focus.metric": {"value": FOCUS, "source": "", "note": "the training-reward axis"},
}
if not LB.empty:
    focus_rows = LB[LB["metric"] == FOCUS]
    for judge in JUDGES:
        jr = focus_rows[focus_rows["judge"] == judge].sort_values("rank_final")
        if jr.empty:
            continue
        top_final = jr.iloc[0]
        top_best = jr.sort_values("rank_best").iloc[0]
        values[f"focus.{judge}.leader_final"] = {
            "value": str(top_final["arm"]), "source": "tables/leaderboard_focus.md",
            "note": (f"highest {FOCUS} at its FINAL state "
                     f"(state {int(top_final['final_state'])}, gain "
                     f"{float(top_final['gain_final']):.3f} vs base)")}
        values[f"focus.{judge}.leader_best"] = {
            "value": str(top_best["arm"]), "source": "tables/leaderboard_focus.md",
            "note": (f"highest {FOCUS} at its BEST state "
                     f"(state {int(top_best['best_state'])}, gain "
                     f"{float(top_best['gain_best']):.3f} vs base)")}
        values[f"focus.{judge}.past_peak_arms"] = {
            "value": sorted(jr[jr["past_peak"]]["arm"].tolist()),
            "source": "tables/leaderboard_focus.md",
            "note": "arms whose best state is not their last -- report both endpoints"}

exports.save_numbers(
    "outcomes", values,
    caption=("Citable headline numbers for this family, each with the table it was read off. "
             "Leaders are per grader and never pooled across graders."))
print(exports.build_index())